In [ ]:
import pandas as pd
import numpy as np
import h5py

# ── 1. EHR 原始数据（chartevents）──────────────────────────────
print("=" * 60)
print("EHR 原始数据（chartevents 前5条心脏病患者记录）")
print("=" * 60)

cardiac = pd.read_csv('/home/featurize/data/output/cardiac_subjects.csv')
static  = pd.read_csv('/home/featurize/data/output/static_features.csv')
stay_ids = set(static['stay_id'].astype(int))

# 只读前几条
for chunk in pd.read_csv(
    '/home/featurize/data/mimic_2.2/icu/chartevents.csv.gz',
    usecols=['stay_id', 'itemid', 'charttime', 'valuenum', 'valueuom'],
    chunksize=1_000_000,
):
    sub = chunk[chunk['stay_id'].isin(stay_ids)]
    if len(sub):
        print(sub.head(10).to_string())
        break

# ── 2. ECG 原始波形数值 ────────────────────────────────────────
print("\n" + "=" * 60)
print("ECG 原始波形数值")
print("=" * 60)

with h5py.File('/home/featurize/data/output/ecg_waveforms.h5', 'r') as f:
    keys = list(f['waveforms'].keys())
    study_id = keys[0]
    sig   = f['waveforms'][study_id]['signal'][:]
    leads = [l.decode() for l in f['waveforms'][study_id]['lead_names'][:]]
    fs    = f['waveforms'][study_id].attrs['fs']

print(f"study_id={study_id}，采样率={fs}Hz，时长={sig.shape[1]//fs}秒")
print(f"\n各导联前20个采样点（单位：mV，归一化后）:")
df_ecg = pd.DataFrame(sig[:, :20], index=leads)
print(df_ecg.round(4).to_string())

print(f"\n各导联统计：")
stats = pd.DataFrame({
    '均值':  sig.mean(axis=1),
    '标准差': sig.std(axis=1),
    '最小值': sig.min(axis=1),
    '最大值': sig.max(axis=1),
}, index=leads)
print(stats.round(4).to_string())

# ── 3. 文本原始数据（出院小结完整内容）──────────────────────────
print("\n" + "=" * 60)
print("出院小结完整原文（心脏病患者第一条，包含所有章节）")
print("=" * 60)

hadm_ids = set(cardiac['hadm_id'].astype(int))
for chunk in pd.read_csv(
    '/home/featurize/data/note/discharge.csv.gz',
    chunksize=10000,
    dtype={'hadm_id': 'Int64'}
):
    sub = chunk[chunk['hadm_id'].isin(hadm_ids)]
    if len(sub):
        print(sub.iloc[0]['text'])
        break

# ── 4. 放射报告原始内容 ───────────────────────────────────────
print("\n" + "=" * 60)
print("放射报告原文（心脏病患者第一条）")
print("=" * 60)

for chunk in pd.read_csv(
    '/home/featurize/data/note/radiology.csv.gz',
    chunksize=10000,
    dtype={'hadm_id': 'Int64'}
):
    sub = chunk[chunk['hadm_id'].isin(hadm_ids)]
    if len(sub):
        print(sub.iloc[0]['text'])
        break

In [ ]:
import pandas as pd

# 只读一行看列名
df = pd.read_csv(
    '/home/featurize/data/mimic_2.2/icu/chartevents.csv.gz',
    nrows=1
)
print("chartevents 所有列名:")
print(df.columns.tolist())
print("\n第一行数据:")
print(df.iloc[0])

In [ ]:
import pandas as pd

lab = pd.read_parquet('/home/featurize/data/output/lab_timeseries.parquet')
print("shape:", lab.shape)
print("列名:", lab.columns.tolist())
print("\n前5行:")
print(lab.head(5).to_string())
print("\nstay_id 数量:", lab['stay_id'].nunique() if 'stay_id' in lab.columns else "无stay_id列")

In [ ]:
import pandas as pd

# ICU 监护指标（chartevents 用的）
d_items = pd.read_csv('/home/featurize/data/mimic_2.2/icu/d_items.csv.gz')

chart_itemids = [
    220045, 220050, 220051, 220052, 220179, 220180, 220181,
    220210, 223761, 220277, 220235, 220224, 220228, 220546,
    220602, 220635, 220645
]

print("=" * 60)
print("ICU 监护变量（chartevents）")
print("=" * 60)
print(d_items[d_items['itemid'].isin(chart_itemids)][['itemid','label','unitname','param_type']].to_string())

# 实验室检查指标（labevents 用的）
d_labitems = pd.read_csv('/home/featurize/data/mimic_2.2/hosp/d_labitems.csv.gz')

lab_itemids = [
    50912, 50971, 50983, 50902, 50882, 51221, 51222,
    51248, 51249, 51250, 51265, 51279, 51301, 50868,
    50931, 51006, 50820
]

print("\n" + "=" * 60)
print("实验室检查变量（labevents）")
print("=" * 60)
print(d_labitems[d_labitems['itemid'].isin(lab_itemids)][['itemid','label','fluid','category']].to_string())


In [ ]:
import pandas as pd

static = pd.read_csv('/home/featurize/data/output/static_features.csv')
print("所有列名:")
print(static.columns.tolist())
print("\n前3行:")
print(static[['subject_id','gender','anchor_age','admission_type',
              'insurance','marital_status','race','los',
              'in_hospital_mortality']].head(3).to_string())
print("\n性别分布:")
print(static['gender'].value_counts())
print("\n住院类型分布:")
print(static['admission_type'].value_counts())
print("\n年龄统计:")
print(static['anchor_age'].describe().round(1))

In [ ]:
import pandas as pd
import pickle
import torch

OUT = "/home/featurize/data/output"

print("="*60)
print("① 数据划分是否一致")
print("="*60)
drim_df = pd.read_csv(f"{OUT}/cardiac_drim.csv")
drim_test_stays = set(drim_df[drim_df.split == "test"]["stay_id"])

with open(f"{OUT}/drfuse_input/ehr_phenotyping_48h_test.pkl", "rb") as f:
    drfuse_test = pickle.load(f)
drfuse_test_stays = set(r["stay_id"] for r in drfuse_test)

print("DRIM   test 样本数:", len(drim_test_stays))
print("DrFuse test 样本数:", len(drfuse_test_stays))
print("完全一致:", drim_test_stays == drfuse_test_stays)
print("交集大小:", len(drim_test_stays & drfuse_test_stays))

print()
print("="*60)
print("② 缺失模态比例")
print("="*60)
te = drim_df[drim_df.split == "test"]
print("DRIM test 集 ECG  缺失率: {:.2%}".format(te["ECG"].isna().mean()))
print("DRIM test 集 TEXT 缺失率: {:.2%}".format(te["TEXT"].isna().mean()))

print()
print("="*60)
print("⑤ EHR 编码器参数量(DRIM侧)")
print("="*60)
import sys
sys.path.insert(0, "/home/featurize/DRIM")
from drim.ehr.models import EHREncoder
enc = EHREncoder(embedding_dim=256, dropout=0.3, input_size=68)
n_params = sum(p.numel() for p in enc.parameters())
print("DRIM 单个 EHREncoder 参数量:", n_params)
print("DRIM 两个(shared+unique)总参数量:", n_params * 2)

print()
print("="*60)
print("检查完成,请把完整输出贴回去")
print("="*60)

In [ ]:
import numpy as np
emb = np.load('/home/featurize/data/output/text_embeddings.npy')
print("shape:", emb.shape)
print("是否有NaN:", np.isnan(emb).any())
print("是否有Inf:", np.isinf(emb).any())
print("数值范围:", emb.min(), "~", emb.max())
print("均值/标准差:", emb.mean(), emb.std())

In [ ]:
import pickle
with open('/home/featurize/data/output/drfuse_input/ehr_phenotyping_48h_train.pkl','rb') as f:
    records = pickle.load(f)
print("X shape:", records[0]['X'].shape)  # 应该是 (48, 68)

In [ ]:
import subprocess

print("="*60)
print("⑤-下半 DrFuse EHR编码器参数量")
print("="*60)
result = subprocess.run(
    ["/environment/miniconda3/envs/drfuse/bin/python", "-c", """
import sys
sys.path.insert(0, '/home/featurize/drfuse')
from models.drfuse import EHRTransformer
enc = EHRTransformer(input_size=68, num_classes=6, d_model=256, n_head=8,
                     n_layers_feat=1, n_layers_shared=1, n_layers_distinct=1,
                     dropout=0.3, max_len=48)
n_params = sum(p.numel() for p in enc.parameters())
print('DrFuse EHRTransformer 总参数量:', n_params)
"""],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("报错:", result.stderr)

print()
print("="*60)
print("③④ DrFuse 评估函数相关代码行")
print("="*60)
result2 = subprocess.run(
    ["grep", "-n", "-E", "PRAUC|average_precision|roc_auc|def.*metric|def compute",
     "/home/featurize/drfuse/models/drfuse_trainer.py"],
    capture_output=True, text=True
)
print(result2.stdout)

print()
print("="*60)
print("请把上面全部输出贴回去")
print("="*60)

In [ ]:
import subprocess, os
import pandas as pd

env = os.environ.copy()
env.pop("MPLBACKEND", None)

print("="*60)
print("⑤-下半 DrFuse EHR编码器参数量")
print("="*60)
result = subprocess.run(
    ["/environment/miniconda3/envs/drfuse/bin/python", "-c", """
import sys
sys.path.insert(0, '/home/featurize/drfuse')
from models.drfuse import EHRTransformer
enc = EHRTransformer(input_size=68, num_classes=6, d_model=256, n_head=8,
                     n_layers_feat=1, n_layers_shared=1, n_layers_distinct=1,
                     dropout=0.3, max_len=48)
n_params = sum(p.numel() for p in enc.parameters())
print('DrFuse EHRTransformer 总参数量:', n_params)
"""],
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.returncode != 0:
    print("报错:", result.stderr[-1500:])

print()
print("="*60)
print("test集各类正样本数(确认评估口径是否可比)")
print("="*60)
df = pd.read_csv('/home/featurize/data/output/cardiac_drim.csv')
te = df[df.split == 'test']
for i in range(6):
    print(f'label_{i} 正样本数:', int(te[f'label_{i}'].sum()))